# Lezione 4 — Data leakage: risultati bellissimi e falsi

**Come si usa questo notebook.** Leggi, esegui, prova l'esercizio prima
della soluzione, chiudi con il progetto. Prerequisito: Lezione 3, che ha
diviso i dati in train/validation/test con un contratto preciso (train per
imparare, validation per scegliere, test una volta sola alla fine) — questo
notebook usa quei tre insiemi e mostra come quel contratto si può rompere
in modi silenziosi.

**Cosa saprai fare alla fine:** riconoscere le tre forme principali di
leakage in qualunque pipeline — tua o di altri — e sapere dove guardare
quando un risultato sembra *troppo* bello.

## Parte 1 — Teoria: cos'è il leakage e perché è l'errore più costoso

**Data leakage**: quando nella fase di addestramento entra informazione che
**in produzione non sarà disponibile** — perché viene dal futuro, dal target,
o dai dati di valutazione. Il risultato è sempre lo stesso: metriche
eccellenti offline, fallimento quando il sistema incontra il mondo reale.

È l'errore più costoso perché **non dà sintomi**: la pipeline gira, i numeri
salgono, tutti sono contenti. Te ne accorgi mesi dopo, in produzione. Casi
celebri ovunque: modelli medici che "predicevano" la malattia da un campo
compilato *dopo* la diagnosi, sistemi di credito addestrati con colonne
disponibili solo a pratica conclusa.

Le tre forme principali:

1. **Target leakage** — una feature contiene (o deriva da) la risposta.
   Esempio: predire l'abbandono di un cliente usando la colonna
   "giorni dalla cancellazione", che esiste solo per chi ha già cancellato.
   Il modello sembra un oracolo: sta solo leggendo la risposta.

2. **Contaminazione train/test** — la stessa informazione compare da
   entrambi i lati della divisione. La causa classica: **duplicati o
   near-duplicates** distribuiti tra train e test — se la Lezione 2 non ha
   già ripulito le righe ripetute (stessa chiave, o stessa entità scritta
   in modo diverso) prima di dividere, alcune di quelle copie finiscono per
   caso una in train e una in test. Il modello "riconosce" in test le righe
   già viste, ed ecco perché l'ordine delle lezioni non è casuale: prima si
   deduplica (Lezione 2), poi si divide (Lezione 3).

3. **Leakage da preprocessing** — la più subdola: calcoli una statistica
   (media per imputare, deviazione per scalare) **su tutti i dati** e poi
   dividi. La statistica del train ora contiene informazione del test.
   Regola ferrea: **prima dividi, poi ogni statistica si calcola solo sul
   train** e si *applica* agli altri insiemi.

Vediamo le prime due con numeri veri:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(11)
n = 400
y = rng.integers(0, 2, n)
onesta = y * 0.8 + rng.normal(0, 1, n)          # feature debole ma legittima
trappola = y + rng.normal(0, 0.05, n)           # derivata dal target ("giorni dalla cancellazione")

def accuratezza(feature):
    X = feature.reshape(-1, 1)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
    return LogisticRegression().fit(X_tr, y_tr).score(X_te, y_te)

print(f'Solo feature onesta      : {accuratezza(onesta):.0%}')
print(f'Con la feature-trappola  : {accuratezza(trappola):.0%}   <- troppo bello per essere vero')

Il 99-100% non è un trionfo: è un allarme. Nessun problema reale con feature
oneste dà quei numeri. **Quando il punteggio sembra troppo bello, la prima
ipotesi è il leakage** — questa intuizione da sola vale la lezione.

Ora la contaminazione da duplicati:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X = rng.normal(0, 1, (200, 3))
y2 = (X[:, 0] + rng.normal(0, 1.2, 200) > 0).astype(int)

def knn_score(Xa, ya):
    X_tr, X_te, y_tr, y_te = train_test_split(Xa, ya, test_size=0.3, random_state=0)
    return KNeighborsClassifier(1).fit(X_tr, y_tr).score(X_te, y_te)

# stessa tabella, ma con il 40% delle righe duplicate PRIMA della divisione
indici = rng.choice(200, 80)
X_dup = np.vstack([X, X[indici]])
y_dup = np.concatenate([y2, y2[indici]])

print(f'Dati unici              : {knn_score(X, y2):.0%}   (stima onesta)')
print(f'Con duplicati nei due lati: {knn_score(X_dup, y_dup):.0%}   (gonfiata: riconosce le righe)')

Il secondo punteggio è più alto **senza che il modello sia migliore**: in
test incontra copie esatte di righe che ha memorizzato in train. Per questo
la deduplicazione va fatta **prima** della divisione, non dopo o insieme:
l'ordine delle operazioni di questo corso (Lezione 2 poi Lezione 3) non è
casuale, è la sequenza che previene esattamente questo problema.

## Parte 2 — Teoria: la regola d'oro del preprocessing

Terza forma, quella che quasi tutti sbagliano almeno una volta. Se imputi la
mediana o scali le feature calcolando le statistiche su **tutti** i dati e
poi dividi, il train "conosce" qualcosa del test.

La regola d'oro, che vale per imputazione, scaling, encoding e qualsiasi
trasformazione appresa dai dati:

> **Prima dividi. Poi: `fit` (calcolo delle statistiche) solo sul train;
> `transform` (applicazione) su train, validation e test.**

Con quantità piccole la differenza numerica può sembrare trascurabile — ed è
per questo che l'errore sopravvive: non "esplode", corrompe in silenzio. La
disciplina conta più dell'entità: statistiche dal solo train, sempre.

## Parte 3 — Esercizio guidato

Questa pipeline sui sensori contiene il leakage da preprocessing:

```python
letture['temperature_c'] = letture['temperature_c'].fillna(
    letture['temperature_c'].median())          # <- mediana su TUTTI i dati
train, test = train_test_split(letture, test_size=0.3, random_state=0)
```

Il tuo compito: riscriverla **senza** leakage — prima la divisione, poi la
mediana calcolata sul solo train e applicata a entrambi — e stampare le due
mediane per vedere la differenza con i tuoi occhi.

**Prova tu:**

In [ ]:
from pathlib import Path

letture = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'environmental_sensor_missing_challenge.csv')

# Scrivi qui la versione corretta:
# 1) dividi PRIMA (train_test_split, test_size=0.3, random_state=0)
# 2) calcola la mediana della temperatura SOLO sul train
# 3) applica quella mediana a train e test
# 4) stampa mediana-globale vs mediana-del-train

letture.head()

### Soluzione spiegata

- la divisione avviene sulla tabella grezza: nessuna statistica è ancora
  stata calcolata, quindi nulla può filtrare;
- `mediana_train` è l'unica statistica ammessa, e viene **applicata** al
  test (mai ricalcolata sul test: in produzione i dati nuovi arrivano uno
  alla volta, non hanno una "loro" mediana);
- la differenza tra le due mediane qui è piccola — ed è proprio il punto:
  il leakage da preprocessing non si vede a occhio, si previene con la
  disciplina dell'ordine.

In [ ]:
grezze = letture.copy()
train, test = train_test_split(grezze, test_size=0.3, random_state=0)
train, test = train.copy(), test.copy()

mediana_globale = grezze['temperature_c'].median()   # quella sbagliata da usare
mediana_train = train['temperature_c'].median()      # l'unica legittima

train['temperature_c'] = train['temperature_c'].fillna(mediana_train)
test['temperature_c'] = test['temperature_c'].fillna(mediana_train)

print(f'Mediana su tutti i dati (vietata): {mediana_globale:.2f}')
print(f'Mediana del solo train (corretta): {mediana_train:.2f}')
assert test['temperature_c'].notna().all()

## Parte 4 — Il progetto: Memory AI Lab, passo 4

Il passo di oggi è un **audit anti-leakage** dei tre insiemi creati nella
Lezione 3 (train/validation/test divisi temporalmente), più la correzione
di una cosa che finora abbiamo fatto "male apposta" per non introdurre
troppi concetti insieme: nelle Lezioni 1-3 l'imputazione di `importance` usava la mediana di
tutti i dati — allora non esistevano ancora gli split, ora sì. Da oggi la
statistica viene dal solo train.

L'audit controlla le tre forme di leakage:
1. nessun `memory_id` in più di un insieme;
2. nessun near-duplicate testuale tra insiemi diversi;
3. ordine temporale rispettato (già garantito dagli assert del passo 3).

In [ ]:
processed = Path('..') / 'datasets' / 'processed'
mem_train = pd.read_csv(processed / 'memory_train.csv')
mem_val = pd.read_csv(processed / 'memory_val.csv')
mem_test = pd.read_csv(processed / 'memory_test.csv')

# 1) identita': nessun id condiviso
id_tr, id_va, id_te = set(mem_train.memory_id), set(mem_val.memory_id), set(mem_test.memory_id)
assert not (id_tr & id_va) and not (id_tr & id_te) and not (id_va & id_te)

# 2) contaminazione testuale: near-duplicates tra insiemi diversi
def testi_norm(frame):
    return set(frame['text'].astype('string').str.strip().str.casefold())

sovrapposti_val = testi_norm(mem_train) & testi_norm(mem_val)
sovrapposti_test = (testi_norm(mem_train) | testi_norm(mem_val)) & testi_norm(mem_test)
print(f'Testi condivisi train/val : {len(sovrapposti_val)}')
print(f'Testi condivisi con test  : {len(sovrapposti_test)}')

In [ ]:
# I template dello storico producono testi identici in periodi diversi:
# righe legittime, ma per un modello che imparera' DAL TESTO sono
# contaminazione. Policy conservativa: restano nel train, escono da val/test.
contaminati = testi_norm(mem_train)
mem_val = mem_val[~mem_val['text'].astype('string').str.strip().str.casefold().isin(contaminati)]
contaminati |= testi_norm(mem_val)
mem_test = mem_test[~mem_test['text'].astype('string').str.strip().str.casefold().isin(contaminati)]

# 3) da oggi: statistiche dal solo train
mediana_train = mem_train['importance'].median()

mem_val.to_csv(processed / 'memory_val.csv', index=False)
mem_test.to_csv(processed / 'memory_test.csv', index=False)

print(f'val dopo audit : {len(mem_val)} memorie')
print(f'test dopo audit: {len(mem_test)} memorie')
print(f'mediana importance del train (da usare ovunque): {mediana_train:.2f}')

Nota il costo: l'audit ha **ridotto** validation e test. Fa male, ma è il
prezzo dell'onestà — meglio meno dati di valutazione che metriche gonfiate.
Questa tensione (quantità vs pulizia della valutazione) ti accompagnerà per
tutto il corso.

## Cosa hai imparato

- Il leakage è informazione **non disponibile in produzione** che entra
  nell'addestramento: metriche belle offline, fallimento reale.
- Le tre forme: **target leakage**, **contaminazione train/test** (spesso
  via duplicati), **preprocessing sui dati completi**.
- Un punteggio troppo bello è un **allarme**, non un successo.
- La regola d'oro: prima dividi; `fit` solo sul train, `transform` ovunque.

## Quiz

1. Un collega predice i guasti di domani con accuratezza 99.5% usando anche
   la colonna "ore di fermo macchina del giorno". Dov'è il trucco?
2. Perché i duplicati vanno rimossi *prima* della divisione e non dopo,
   insieme, dentro ogni singolo insieme?
3. Nel progetto abbiamo tolto i testi contaminati da val/test invece che dal
   train. Perché non il contrario?

<details>
<summary><b>Apri le risposte</b></summary>

1. "Ore di fermo del giorno" si conosce solo *dopo* che il guasto è
   avvenuto: è target leakage. In produzione, al momento della previsione,
   quella colonna non esiste ancora.
2. Deduplicare dentro ogni insieme separatamente lascia intatte le coppie
   che stanno una in train e una in test — che sono esattamente quelle che
   contaminano. Il confronto va fatto sull'insieme completo, prima del
   taglio.
3. Il train può permettersi rumore; validation e test sono gli strumenti di
   misura e devono restare puliti. Togliere dal train ridurrebbe
   l'apprendimento senza ripulire la misura: i testi resterebbero comunque
   noti al modello.
</details>

## Fonti

- scikit-learn, *Common pitfalls: data leakage* (definizione e regola
  fit-sul-solo-train): https://scikit-learn.org/stable/common_pitfalls.html
- scikit-learn, *Pipelines* (lo strumento che rende strutturale la regola
  d'oro): https://scikit-learn.org/stable/modules/compose.html

## Prossima lezione

I dati sono divisi e senza perdite. Ma un modello non mangia parole come
"episodic" né numeri su scale diverse: servono **encoding e scaling** — le
ultime trasformazioni prima dei tensori. Lezione 5.